# 03 RFI statistics from OpenSAR

The RFI statistics are derived from the OpenSAR data in a separate analysis
(Google Colab), which is not part of this repository. Its output used here is
`data/processed/opensar_rfi_stats.csv` (not tracked): one row per patch with
`patch_id, split, extent, severity_db_vv, severity_db_vh`. `severity_db_*` is
the measured excess power (dB) of the masked RFI pixels against the unmasked
pixels, from SLC power per polarisation. The `sir_db_*` columns of the file
are not used (they hold a capped value for contrasts of 0.1 dB or less);
SIR is recomputed as SIR = -10 log10(10^(C/10) - 1) for C > 0.1 dB.

Notes: the statistics come from 125 of 223 OpenSAR patches (98 had no
measurable RFI in the mask). They are measured on SLC pixels, while the
injection acts on station-level GRD backscatter. Some patches show no power
excess in one polarisation. If the file is absent, the provisional grid is
used with a warning.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

In [ ]:
from src.rfi import grid_table, injection_grid, load_opensar_stats

stats = load_opensar_stats()
print("OpenSAR statistics:", "not available (provisional grid)" if stats is None
      else f"{len(stats)} patches")
grid = injection_grid()
grid_table(grid).round(4)

## Distribution summary (aggregated)

In [ ]:
if stats is not None:
    display(stats[["extent", "severity_db_vv", "severity_db_vh"]].describe().round(3))
    for p in ("vv", "vh"):
        c = stats[f"severity_db_{p}"]
        print(p.upper(), "patches with contrast <= 0.1 dB:", int((c <= 0.1).sum()),
              "(of which <= 0:", int((c <= 0).sum()), ")")
    print("patches with extent > 0.85:", int((stats["extent"] > 0.85).sum()))